In [ ]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.data_module.data_split import create_splits
import numpy as np

# Brief description about dataset and loader

## Dataset
The dataset collects fetches all the different data sources and does some precomputation, such that they can be loaded efficiently later.

In particular, it caches stationary variables, such that they are already in memory.

Dynamic variables were already rechunked in time, so time-slices can be read in with minimal effort.

In the end the dataset provides a dictionary with 
    - global information (NWP + TOPO at low resolution)
    - observations (measurements at a station + some high res topography info)
    - query points (some high res topography info + (optional) measurements required for validation/testing)

## Dataloader

The dataloader similar to the default pytorch loader, but additional applies some data transforms (e.g. log or scaling) on the fly. Transforms can be defined in the config.yaml file.

For different settings see `config/data/wind_data.yaml`.

## Things that need to be done still:

- How should the stations be split between observations/queries and train/val/test
- Can validation condition on training stations? And test on training and validation?
- Dynamically set query points?
- At the moment cached is saved where the code is executed. You want to specify that.
- Anything else?


In [ ]:
from torch.utils.data import DataLoader
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))
train_dataset, val_dataset, test_dataset = create_splits(dataset)


# Batch Structure

| Data Group | Field | Shape | Description | Paper Notation |
|------------|-------|-------|-------------|----------------|
| **global_data** | global_field | [B, C, H_lr, W_lr] | Coarse NWP fields + low-res topography | U_nwp |
| | time_emb | [B, C_time] | Time features: sin/cos(diurnal, annual) | τ |
| **observation_data (KV)** | coords | [B, N_kv, 2] | Station coordinates on Swiss grid | x_kv |
| | geo | [B, N_kv, C_geo] | Local geometry: position, elevation | g_kv |
| | topo | [B, N_kv, C_topo, H_hr, W_hr] | High-res topographic patches | ξ_kv |
| | obs | [B, N_kv, C_y] | Station measurements | y_kv |
| | mask_obs | [B, N_kv, C_y] | Binary mask for valid observations | M_obs |
| **query_data** | coords | [B, N_q, 2] | Query point coordinates | x_q |
| | geo | [B, N_q, C_geo] | Local geometry features | g_q |
| | topo | [B, N_q, C_topo, H_hr, W_hr] | High-res topographic patches | ξ_q |
| | obs | [B, N_q, C_y] | (Optional) Measurements at query points | - |
| | mask_obs | [B, N_q, C_y] | (Optional) Observation mask | - |

---

### Notation Mapping
- **B**: Batch size
- **N_kv**: Number of observation (station) points
- **N_q**: Number of query points
- **C, C_y, C_geo, C_topo, C_time**: Channel dimensions
- **H_lr, W_lr**: Low-resolution spatial dimensions
- **H_hr, W_hr**: High-resolution patch dimensions

## Global Data

In [ ]:
from matplotlib import pyplot as plt

global_vars = data_config.vars.nwp_vars + data_config.vars.topo_vars
data = train_dataset[0]
for chan_idx, global_var in enumerate(global_vars):
    plt.pcolor(data['global_data']['global_field'][chan_idx].cpu(), cmap='bwr')
    plt.colorbar(label=global_var)
    plt.show()

In [ ]:
plt.scatter(data['query_data']['coords'][:,0].cpu(), data['query_data']['coords'][:,1].cpu(), c='blue', label='Queries')
plt.scatter(data['observation_data']['coords'][:,0].cpu(), data['observation_data']['coords'][:,1].cpu(), c='red', label='Observations')
plt.xlabel('Swiss Grid Easting')
plt.ylabel('Swiss Grid Northing')
plt.title('Query Data Coordinates')
plt.legend()
plt.show()

In [ ]:
for chan_idx, variable in enumerate(data_config.vars.obs_vars):
    plt.scatter(data['observation_data']['coords'][:,0].cpu(), data['observation_data']['coords'][:,1].cpu(), c=data['observation_data']['obs'][:, chan_idx].cpu(), cmap='inferno')
    plt.colorbar(label=variable)
    plt.xlabel('Swiss Grid Easting')
    plt.ylabel('Swiss Grid Northing')
    plt.title('Observation Data Coordinates with Values')
plt.show()

## Topo data around station

### At observed station

In [ ]:
# For observation
station_idx = 10
dem_idx = np.where(np.array(data_config.vars.topo_vars) == 'DEM')[0][0]
plt.pcolor(data['observation_data']['topo'][station_idx, dem_idx].cpu(), cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.title('Topography')

### At query point

In [ ]:
station_idx = 10
dem_idx = np.where(np.array(data_config.vars.topo_vars) == 'DEM')[0][0]
plt.pcolor(data['query_data']['topo'][station_idx, dem_idx].cpu(), cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.title('Topography')